# Bakken Den Haag
https://den-haag-opendata.opendatasoft.com/explore/assets/bakken/

Beschrijving: Locaties van ondergrondse en bovengrondse afval apartplaatsen (inclusief Oracs) en zoutafhaalkisten.
Bron: beheersysteem gemeente Den Haag
Doel registratie: Beheer openbare ruimte
Coördinatenstelsel: RDnew. De KML is WGS84

https://den-haag-opendata.opendatasoft.com/explore/assets/wijken/
Beschrijving: Wijkgrenzen van Den Haag
Bron: gemeente Den Haag
Doel registratie: Bestuurlijke grenzen
Coördinatenstelsel: RDnew. De KML is WGS84

## Data
- `bakken`: GeoDataFrame of Den Haag waste containers (CRS: WGS84/OGC:CRS84)
  - Key columns: `afvalfractie_code` (e.g. `"PPR"` = paper), `wijk_code` (str), `geometry` (points)
- `wijken`: GeoDataFrame of Den Haag neighbourhood borders
  - Key columns: `wijkcode` (int), `wijknaam` (str), `geometry` (polygons), `geo_point_2d` (centroids — default/active geometry, switch with `set_geometry('geometry')`)
  - CRS: OGC:CRS84

## Gotchas
- `wijken.geometry` is active but contains centroids; polygons are in the `geometry` column — always `set_geometry('geometry').set_crs('OGC:CRS84')` before use
- `wijken['wijkcode']` is `int64`; `bakken['wijk_code']` is `str`
- Reproject to EPSG:3857 for contextily basemaps and metric distances

In [ ]:
import pandas as pd
import geopandas as gpd

import numpy as np
import osmnx as ox
import networkx as nx
import shapely
import contextily as cx
import matplotlib.pyplot as plt

In [ ]:
wijken = gpd.read_parquet('../data/wijken.parquet')
bakken = gpd.read_parquet('../data/bakken.parquet')

In [ ]:
# --- Setup (same as before) ---
wijken_poly = wijken.set_geometry('geometry').set_crs('OGC:CRS84')
laak = wijken_poly[wijken_poly['wijkcode'] == 38].to_crs(epsg=3857)
laak_poly = laak.union_all()

paper_wm = bakken[bakken['afvalfractie_code'] == "PPR"].to_crs(epsg=3857)

xmin, ymin, xmax, ymax = laak.total_bounds
res = 2500
gx = np.linspace(xmin, xmax, res)
gy = np.linspace(ymin, ymax, res)
gxx, gyy = np.meshgrid(gx, gy)
mask = shapely.contains_xy(laak_poly, gxx.ravel(), gyy.ravel()).reshape(res, res)

# --- Road network ---
# Buffer the polygon so roads crossing the boundary are intact
buffer_m = 600
laak_buffered = laak.buffer(buffer_m).union_all()
laak_buffered_wgs = gpd.GeoSeries([laak_buffered], crs='epsg:3857').to_crs(4326).iloc[0]

G = ox.graph_from_polygon(laak_buffered_wgs, network_type='walk')
G = ox.project_graph(G, to_crs='epsg:3857')

# Snap bins to nearest network nodes, add snap offset to distances
# Use ALL paper bins within the buffer, not just those clipped to laak
paper_buffered = bakken[bakken['afvalfractie_code'] == "PPR"].to_crs(epsg=3857)
paper_buffered = paper_buffered[paper_buffered.within(laak_buffered)]
coords = np.array([[g.x, g.y] for g in paper_buffered.geometry])
bin_node_ids, bin_snap_dists = ox.distance.nearest_nodes(
    G, coords[:, 0], coords[:, 1], return_dist=True
)

# Multi-source Dijkstra seeded with snap offsets so bin-to-node
# distance is accounted for from the start
sources = {}
for node, snap_d in zip(bin_node_ids, bin_snap_dists):
    if node not in sources or snap_d < sources[node]:
        sources[node] = snap_d
length_map = nx.multi_source_dijkstra_path_length(
    G, sources, weight='length'
)

# Snap grid points to nearest node, add snap offset, keep only points
# whose snapped node is inside (or very close to) the wijk polygon
grid_nodes, grid_snap_dists = ox.distance.nearest_nodes(
    G, gxx.ravel(), gyy.ravel(), return_dist=True
)

In [ ]:
# Build a dense point cloud from all network nodes that have a distance
node_xs = np.array([G.nodes[n]['x'] for n in length_map])
node_ys = np.array([G.nodes[n]['y'] for n in length_map])
node_ds = np.array([length_map[n] for n in length_map])

# Interpolate node distances onto the fine grid using linear interpolation
from scipy.interpolate import LinearNDInterpolator
interp = LinearNDInterpolator(np.c_[node_xs, node_ys], node_ds)
dist_road = interp(gxx.ravel(), gyy.ravel())
dist_grid = np.where(mask, dist_road.reshape(res, res), np.nan)

In [ ]:
# --- Plot ---
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

# Segmented colormap with gradients within each isochrone band
cmap = mcolors.LinearSegmentedColormap.from_list(
    'isochrone',
    [
        (0/500,    '#1a9641'),  # dark green   – 0 m
        (125/500,  '#a6d96a'),  # light green  – 125 m
        (250/500,  '#fdae61'),  # orange       – 250 m
        (375/500,  '#d7191c'),  # red          – 375 m
        (500/500,  '#7b0000'),  # dark red     – 500 m
    ]
)
norm = mcolors.Normalize(vmin=0, vmax=500)

fig, ax = plt.subplots(figsize=(16, 12))

# Isochrone fill
ax.imshow(
    dist_grid,
    origin='lower',
    extent=[xmin, xmax, ymin, ymax],
    cmap=cmap,
    norm=norm,
    alpha=0.6,
    aspect='auto',
    zorder=2
)

# Isochrone contour lines
cs = ax.contour(
    gxx, gyy, dist_grid,
    levels=[125, 250, 375, 500],
    colors=['#1a9641', '#f4d03f', '#e67e22', '#d7191c'],
    linewidths=1.5,
    zorder=3
)

laak.plot(ax=ax, color='none', edgecolor='black', linewidth=2, zorder=4)
paper_buffered.cx[xmin:xmax, ymin:ymax].plot(
    ax=ax, color='blue', markersize=5, zorder=5, label='Papierbak'
)
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zorder=1)

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
cb = plt.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    ax=ax, label='Loopafstand (m)', shrink=0.7
)
cb.set_ticks([0, 125, 250, 375, 500])
cb.set_ticklabels(['0 m', '125 m\n(norm)', '250 m\n(2× norm)', '375 m\n(3× norm)', '500 m\n(4× norm)'])
ax.set_title('Loopafstand tot dichtstbijzijnde papierbak (Laakkwartier en Spoorwijk)')
ax.set_axis_off()

# Legend for isochrones
ax.legend(
    handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue',
                        markersize=6, label='Papierbak')],
    loc='lower right', fontsize=9
)

plt.tight_layout()
plt.savefig("../img/afval_container_papier_loopafstand_norm.png", dpi=300)
plt.show()